# Logistic Regression: Fake News Detection with ML

Objective: An automatic learning system capable of predicting whether a given news article is **fake** or **real**. Our data is **text**. ML algorithms work with numbers, not words. Therefore, we will have to **transform text into numbers** before we can train our model. This process is called **vectorization**.

### Dataset

##### [Fake and Real News Dataset](https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset)

This dataset contains news collected from various sources and consists of **two CSV files**:

- `True.csv`: contains **true** (real) news.
- `Fake.csv`: contains **fake** news.

Each file has the following columns:

| Column  | Description               |
|---------|---------------------------|
| `title` | News title                |
| `text`  | Full body/content of news |
| `subject` | News thematic category    |
| `date`  | Publication date          |

In total, the dataset contains over **44,000 news articles**, approximately half fake and half real.

**Download**: You can download the dataset from [Kaggle](https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset). Once downloaded, place the `True.csv` and `Fake.csv` files in the same directory as this notebook.

### 1. Reading the dataset

In [ ]:
# IMPORTANT: Run this cell to install dependencies before starting
!pip install pandas beautifulsoup4 nltk scikit-learn

In [ ]:
import pandas as pd

In [ ]:
# Read the two CSV files
df_true = pd.read_csv("True.csv")
df_fake = pd.read_csv("Fake.csv")

In [ ]:
# Explore the first rows of the true news dataset
df_true.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [ ]:
# Explore the first rows of the fake news dataset
df_fake.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [ ]:
# Add a "label" column to each DataFrame to identify the news
df_true["label"] = "REAL"
df_fake["label"] = "FAKE"

In [ ]:
# Join both DataFrames into one
df = pd.concat([df_true, df_fake], ignore_index=True)

In [ ]:
# Verify the result
print(f"True news: {len(df_true)}")
print(f"Fake news: {len(df_fake)}")
print(f"Total news: {len(df)}")

True news: 21417
Fake news: 23481
Total news: 44898


In [ ]:
df.head()

,title,text,subject,date,label
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",REAL
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",REAL
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",REAL
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017",REAL
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017",REAL


In [ ]:
df.tail()

,title,text,subject,date,label
44893,McPain: John McCain Furious That Iran Treated ...,21st Century Wire says As 21WIRE reported earl...,Middle-east,"January 16, 2016",FAKE
44894,JUSTICE? Yahoo Settles E-mail Privacy Class-ac...,21st Century Wire says It s a familiar theme. ...,Middle-east,"January 16, 2016",FAKE
44895,Sunnistan: US and Allied ‘Safe Zone’ Plan to T...,Patrick Henningsen 21st Century WireRemember ...,Middle-east,"January 15, 2016",FAKE
44896,How to Blow $700 Million: Al Jazeera America F...,21st Century Wire says Al Jazeera America will...,Middle-east,"January 14, 2016",FAKE
44897,10 U.S. Navy Sailors Held by Iranian Military ...,21st Century Wire says As 21WIRE predicted in ...,Middle-east,"January 12, 2016",FAKE


### 2. Dataset Exploration

In [ ]:
# Distribution of news by label
df["label"].value_counts()

,count
label,
FAKE,23481
REAL,21417


In [ ]:
# Let's see an example of a REAL news article
print("=" * 80)
print("REAL NEWS")
print("=" * 80)
print(f"\nTitle: {df[df['label']=='REAL'].iloc[0]['title']}")
print(f"\nText: {df[df['label']=='REAL'].iloc[0]['text'][:500]}...")

REAL NEWS

Title: As U.S. budget fight looms, Republicans flip their fiscal script

Text: WASHINGTON (Reuters) - The head of a conservative Republican faction in the U.S. Congress, who voted this month for a huge expansion of the national debt to pay for tax cuts, called himself a “fiscal conservative” on Sunday and urged budget restraint in 2018. In keeping with a sharp pivot under way among Republicans, U.S. Representative Mark Meadows, speaking on CBS’ “Face the Nation,” drew a hard line on federal spending, which lawmakers are bracing to do battle over in January. When they retur...


In [ ]:
# Let's see an example of a FAKE news article
print("=" * 80)
print("FAKE NEWS")
print("=" * 80)
print(f"\nTitle: {df[df['label']=='FAKE'].iloc[0]['title']}")
print(f"\nText: {df[df['label']=='FAKE'].iloc[0]['text'][:500]}...")

FAKE NEWS

Title:  Donald Trump Sends Out Embarrassing New Year’s Eve Message; This is Disturbing

Text: Donald Trump just couldn t wish all Americans a Happy New Year and leave it at that. Instead, he had to give a shout out to his enemies, haters and  the very dishonest fake news media.  The former reality show star had just one job to do and he couldn t do it. As our Country rapidly grows stronger and smarter, I want to wish all of my friends, supporters, enemies, haters, and even the very dishonest Fake News Media, a Happy and Healthy New Year,  President Angry Pants tweeted.  2018 will be a gr...


**Observation**: If you look at the fake news, many of them contain **remnants of HTML code**, URLs, special characters, and other elements that are not part of the actual news text. We need to **clean** this text.

### 3. Text Preprocessing

Here we encounter one of the **big challenges** of Machine Learning applied to text: the data does not come clean. News articles can contain:

- **HTML code** (tags like `<p>`, `<br>`, `<div>`, etc.)
- **URLs** (web links)
- **Punctuation marks** that do not provide useful information
- **Very common words** that appear in all news articles and do not help distinguish between fake and real (e.g., "the", "is", "and", "a"...)

Before we can train our model, we need to **process and clean the text** to keep only the relevant information. Let's do it step by step.

#### 3.1. Removing HTML code

Many news articles, especially those collected from the internet, may contain remnants of HTML code. We need to keep only the text and remove all HTML tags (BeautifulSoup).

In [ ]:
import warnings
from bs4 import BeautifulSoup, MarkupResemblesLocatorWarning

warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)

In [ ]:
# Example of text with HTML
html_example = '<p>This is a <b>breaking</b> news <a href="http://example.com">story</a></p>'
print("Original text:", html_example)

Original text: <p>This is a <b>breaking</b> news <a href="http://example.com">story</a></p>


In [ ]:
# Remove HTML tags
clean_text = BeautifulSoup(html_example, "html.parser").get_text()
print("Clean text:", clean_text)

Clean text: This is a breaking news story


In [ ]:
def strip_html(text):
    """Removes HTML tags from a text."""
    return BeautifulSoup(text, "html.parser").get_text()

In [ ]:
# Test with a real news article from the dataset
print(strip_html(df.iloc[0]["text"])[:300])

WASHINGTON (Reuters) - The head of a conservative Republican faction in the U.S. Congress, who voted this month for a huge expansion of the national debt to pay for tax cuts, called himself a “fiscal conservative” on Sunday and urged budget restraint in 2018. In keeping with a sharp pivot under way 


#### 3.2. Removing URLs

News articles may also contain URLs (web addresses) that do not provide useful information for our classifier. We need to remove them (regular expressions).

In [ ]:
import re

In [ ]:
# Example of text with URL
url_example = "Breaking news about the economy https://www.example.com/news check it out"
print("Original text:", url_example)

Original text: Breaking news about the economy https://www.example.com/news check it out


In [ ]:
# Remove URLs
text_without_urls = re.sub(r'https?://\S+', '', url_example)
print("Clean text:", text_without_urls)

Clean text: Breaking news about the economy  check it out


In [ ]:
def remove_urls(text):
    """Removes URLs from a text."""
    return re.sub(r'https?://\S+', '', text)

#### 3.3. Removing punctuation and converting to lowercase

Punctuation marks (commas, periods, exclamation marks, etc.) generally do not provide useful information for distinguishing between fake and real news. In addition, we want our algorithm to treat words the same regardless of whether they are capitalized or lowercase (for example, "President" and "president" should be the same word) (using the `.lower()` method and the `string` library).

In [ ]:
import string

# These are all the punctuation marks Python recognizes
print(string.punctuation)

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~


In [ ]:
def remove_punctuation(text):
    """Converts text to lowercase and removes punctuation."""
    text = text.lower()
    # Remove standard ASCII punctuation - classic but incomplete
    # text = text.translate(str.maketrans('', '', string.punctuation))
    # Using RE
    text = re.sub(r'[^\w\s]', '', text, flags=re.UNICODE)
    return text

In [ ]:
# Example
example_text = "BREAKING NEWS! The President said: 'No comment'."
print("Original:", example_text)
print("Processed:", remove_punctuation(example_text))

Original: BREAKING NEWS! The President said: 'No comment'.
Processed: breaking news the president said no comment


#### 3.4. Removing stopwords and Stemming

Now we are going to apply two fundamental techniques of **Natural Language Processing (NLP)**:

**Stopwords**: These are very common words in a language that appear in almost all texts and, therefore, do not provide useful information for distinguishing between categories. Some examples are: *"the"*, *"is"*, *"and"*, *"a"*, *"in"*, *"to"*...

Imagine you are trying to distinguish if a news article is fake or real. The word *"the"* will appear in both types of news with a similar frequency, so it doesn't help us. However, words like *"hoax"* or *"verified"* could be more informative (list of stopwords: nltk.corpus.stopwords.words('english')).

**Stemming**: It is the process of reducing a word to its **root**. For example:
- *"running"*, *"runs"*, *"ran"* → **"run"**
- *"playing"*, *"played"*, *"plays"* → **"play"**

Why is this important to us? Because for our algorithm, *"running"* and *"runs"* are completely different words, when in reality they have the same basic meaning. By reducing them to their root, we group the variants of the same word (we use the nltk library).

In [ ]:
import nltk

# Download necessary nltk resources (only the first time)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

True

In [ ]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

In [ ]:
# Let's see some English stopwords
stop_words = set(stopwords.words('english'))
print(f"Number of stopwords: {len(stop_words)}")
print(f"\nSome examples: {list(stop_words)[:20]}")

Number of stopwords: 198

Some examples: ['up', "i'll", 'he', 'some', 'and', 's', 'who', 'here', "he's", 'both', "hasn't", 'my', 'very', 'those', "couldn't", "we'll", "you'll", 'were', 'again', "doesn't"]


In [ ]:
# Let's see how the Stemmer works
stemmer = PorterStemmer()

words_to_stem = ["running", "runs", "ran", "playing", "played", "investigation", "investigating"]
for word in words_to_stem:
    print(f"  {word:20s} → {stemmer.stem(word)}")

  running              → run
  runs                 → run
  ran                  → ran
  playing              → play
  played               → play
  investigation        → investig
  investigating        → investig


In [ ]:
# Full example: tokenize, remove stopwords, and apply stemming
example_sentence = "The president was running an investigation into the reported claims"
print("Original text:", example_sentence)

# 1. Tokenize (split into words)
tokens = word_tokenize(example_sentence.lower())
print("\nTokens:", tokens)

# 2. Remove stopwords
filtered_tokens = [t for t in tokens if t not in stop_words]
print("\nWithout stopwords:", filtered_tokens)

# 3. Apply stemming
stemmed_tokens = [stemmer.stem(t) for t in filtered_tokens]
print("\nWith stemming:", stemmed_tokens)

Original text: The president was running an investigation into the reported claims

Tokens: ['the', 'president', 'was', 'running', 'an', 'investigation', 'into', 'the', 'reported', 'claims']

Without stopwords: ['president', 'running', 'investigation', 'reported', 'claims']

With stemming: ['presid', 'run', 'investig', 'report', 'claim']


### 4. Complete preprocessing function

Now we create a function that processes a news article from beginning to end, applying all transformations (HTML removal, URL removal, lowercase conversion, punctuation removal, stopwords removal, and stemming) and returns the processed text as a single string.

In [ ]:
def preprocess_text(text):
    """
    Applies all preprocessing transformations to a text:
    0. Remove source prefixes
    1. Remove HTML tags
    2. Remove URLs
    3. Convert to lowercase
    4. Remove punctuation
    5. Tokenize
    6. Remove stopwords
    7. Apply stemming

    Returns the processed text as a single string.
    """
    text = re.sub(r'^[A-Z\s,.]+ \([^)]+\)\s*[-–—]?\s*', '', text)
    text = strip_html(text)
    text = remove_urls(text)
    text = remove_punctuation(text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stop_words]
    stemmer = PorterStemmer()
    tokens = [stemmer.stem(t) for t in tokens]

    # Join tokens back into a string
    return " ".join(tokens)

In [ ]:
# Test with a news article from the dataset
original_text = df.iloc[0]["text"]
print("ORIGINAL TEXT (first 300 characters):")
print(original_text[:300])
print("\n" + "=" * 80)
print("\nPROCESSED TEXT (first 300 characters):")
print(preprocess_text(original_text)[:300])

ORIGINAL TEXT (first 300 characters):
WASHINGTON (Reuters) - The head of a conservative Republican faction in the U.S. Congress, who voted this month for a huge expansion of the national debt to pay for tax cuts, called himself a “fiscal conservative” on Sunday and urged budget restraint in 2018. In keeping with a sharp pivot under way 


PROCESSED TEXT (first 300 characters):
head conserv republican faction us congress vote month huge expans nation debt pay tax cut call fiscal conserv sunday urg budget restraint 2018 keep sharp pivot way among republican us repres mark meadow speak cb face nation drew hard line feder spend lawmak brace battl januari return holiday wednes


The processed text is much cleaner and contains only the **relevant words reduced to their root**. This is the text that our Machine Learning algorithm will receive as input.

### 5. Applying preprocessing to the dataset

Now that we have our preprocessing function, let's apply it to all news articles in the dataset. This process may take a while as we have over 44,000 news articles (we start by applying it to a sample of 1000 news articles).

In [ ]:
# To start, we work with a subset of 1000 news articles
# Shuffle the DataFrame to have mixed fake and real news
df_sample = df.sample(n=1000, random_state=42)

print(f"Subset size: {len(df_sample)}")
print(f"\nLabel distribution:")
print(df_sample["label"].value_counts())

Subset size: 1000

Label distribution:
label
FAKE    536
REAL    464
Name: count, dtype: int64


In [ ]:
# Apply preprocessing to all news articles in the subset
# This may take a few seconds
print("Preprocessing news...")
df_sample["text_clean"] = df_sample["text"].apply(preprocess_text)
print("Done!")

Preprocessing news...
Done!


In [ ]:
# Let's see the result
df_sample[["text", "text_clean", "label"]].head()

,text,text_clean,label
22216,"Donald Trump s White House is in chaos, and th...",donald trump white hous chao tri cover russia ...,FAKE
27917,Now that Donald Trump is the presumptive GOP n...,donald trump presumpt gop nomine time rememb c...,FAKE
25007,Mike Pence is a huge homophobe. He supports ex...,mike penc huge homophob support exgay convers ...,FAKE
1377,SAN FRANCISCO (Reuters) - California Attorney ...,california attorney gener xavier becerra said ...,REAL
32476,Twisted reasoning is all that comes from Pelos...,twist reason come pelosi day especi 2006 promi...,FAKE


In [ ]:
# Example of before and after
print("BEFORE:")
print(df_sample.iloc[0]["text"][:200])
print("\nAFTER:")
print(df_sample.iloc[0]["text_clean"][:200])

BEFORE:
Donald Trump s White House is in chaos, and they are trying to cover it up. Their Russia problems are mounting by the hour, and they refuse to acknowledge that there are problems surrounding all of th

AFTER:
donald trump white hous chao tri cover russia problem mount hour refus acknowledg problem surround fake news hoax howev fact bear thing differ seem crack congression public leadershipchuck grassley ri


### 6. Encoding text: vectorization

Our data is **text** and Machine Learning algorithms need to receive **numbers**. **Vectorization** is converting text to numbers, and there are multiple techniques. We will use one of the simplest: **Bag of Words**.

#### How does Bag of Words work?

1. **Create a vocabulary** with all unique words that appear in the entire dataset: `["law", "new", "president", "results", "shows", "signs", "study", "today"]`

2. **Represent each news article as a vector** of numbers, where each number indicates **how many times each word appears** in that news article:

| | law | new | president | results | shows | signs | study | today |
|---|---|---|---|---|---|---|---|---|
| **News 1** | 1 | 1 | 1 | 0 | 0 | 1 | 0 | 1 |
| **News 2** | 0 | 1 | 0 | 1 | 1 | 0 | 1 | 1 |

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

### 6.1 CountVectorizer small example

In [74]:
sample_texts = [
    "president signs new law today",
    "new study shows results today"
]

vectorizer = CountVectorizer() # Uses Bag of Words
vectorizer.fit(sample_texts) # Learns the vocabulary

print("Learned vocabulary:")
print(vectorizer.get_feature_names_out())

Learned vocabulary:
['law' 'new' 'president' 'results' 'shows' 'signs' 'study' 'today']


In [76]:
# Transforming text into vectors
vector = vectorizer.transform(sample_texts) # Transform news text into vectors

# Visualize the result matrix
pd.DataFrame(
    vector.toarray(),
    columns=vectorizer.get_feature_names_out(),
    index=["News 1", "News 2"]
)

,law,new,president,results,shows,signs,study,today
News 1,1,1,1,0,0,1,0,1
News 2,0,1,0,1,1,0,1,1


### 6.2 Apply to the real dataset

In [ ]:
# Apply CountVectorizer on our processed texts
vectorizer = CountVectorizer()
vectorizer.fit(df_sample["text_clean"])

print(f"Vocabulary size: {len(vectorizer.get_feature_names_out())} unique words")

Vocabulary size: 19185 unique words


In [ ]:
# Transform the texts into vectors
X_vect = vectorizer.transform(df_sample["text_clean"])

print(f"Dimensions of the resulting matrix: {X_vect.shape}")
print(f"  → {X_vect.shape[0]} news articles")
print(f"  → {X_vect.shape[1]} words in the vocabulary")

Dimensions of the resulting matrix: (1000, 19185)
  → 1000 news articles
  → 19185 words in the vocabulary


In [ ]:
# Let's see the first rows of the matrix (only the first 10 columns)
pd.DataFrame(
    X_vect.toarray(),
    columns=vectorizer.get_feature_names_out()
)

,000004,00009,0006,005380k,0100,015760k,020,0259,0300,0307,...,zip,zipper,zippi,zolani,zombi,zone,zoran,zuckerberg,zuma,zweli
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
996,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
997,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
998,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


The resulting matrix is very big (each news is represented by a vector of thousands of elements) and most of the values are 0 (an individual news contains only a small fraction of all words in the vocabulary). These types of matrices are called **sparse matrices** and scikit-learn handles them efficiently internally.

### 7. Algorithm Training

We are going to train a **Logistic Regression** model that will learn from these vectors to classify news as fake or real. We provide it with training data (news with their labels) and the algorithm learns the patterns that distinguish a fake news from a real one.

In [ ]:
# We read only a subset of 1500 news articles
df_all = df.sample(n=1500, random_state=42)

# We keep 1000 news articles for training
df_train_sample = df_all.iloc[:1000]

print(f"Subset size: {len(df_train_sample)}")
print(f"\nLabel distribution:")
print(df_train_sample["label"].value_counts())

Subset size: 1000

Label distribution:
label
FAKE    536
REAL    464
Name: count, dtype: int64


In [ ]:
# Apply preprocessing
print("Preprocessing news...")
df_train_sample["text_clean"] = df_train_sample["text"].apply(preprocess_text)
print("Done!")

Preprocessing news...
Done!


/tmp/ipykernel_7177/62170797.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train_sample["text_clean"] = df_train_sample["text"].apply(preprocess_text)


In [ ]:
# Vectorize the preprocessed text
vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(df_train_sample["text_clean"])

In [ ]:
print(X_train.toarray())
print("\nFeatures:", len(vectorizer.get_feature_names_out()))

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]

Features: 19185


In [ ]:
pd.DataFrame(X_train.toarray(), columns=[vectorizer.get_feature_names_out()])

,000004,00009,0006,005380k,0100,015760k,020,0259,0300,0307,...,zip,zipper,zippi,zolani,zombi,zone,zoran,zuckerberg,zuma,zweli
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
996,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
997,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
998,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
y_train = df_train_sample["label"]
y_train

,label
22216,FAKE
27917,FAKE
25007,FAKE
1377,REAL
32476,FAKE
...,...
27000,FAKE
25165,FAKE
20813,REAL
11756,REAL


In [ ]:
# Train the Logistic Regression algorithm
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

### 8. Prediction

##### Reading a set of new news

In [ ]:
# Take the 500 news articles that have NOT been used to train the algorithm
df_test = df_all.iloc[1000:]

print("Preprocessing test news...")
df_test["text_clean"] = df_test["text"].apply(preprocess_text)
print("Done!")

Preprocessing test news...
Done!


/tmp/ipykernel_7177/2479274878.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["text_clean"] = df_test["text"].apply(preprocess_text)


In [ ]:
X_test = df_test["text_clean"]
y_test = df_test["label"]

print(f"Noticias de test: {len(X_test)}")

Noticias de test: 500


##### Preprocessing news with the vectorizer

In [ ]:
# Apply CountVectorizer (solo .transform())
X_test = vectorizer.transform(X_test)

##### Predicting the news type

In [ ]:
y_pred = clf.predict(X_test)
y_pred

array(['REAL', 'REAL', 'FAKE', 'FAKE', 'REAL', 'FAKE', 'FAKE', 'FAKE',
       'FAKE', 'FAKE', 'REAL', 'REAL', 'FAKE', 'FAKE', 'FAKE', 'REAL',
       'REAL', 'REAL', 'REAL', 'FAKE', 'REAL', 'REAL', 'REAL', 'FAKE',
       'REAL', 'REAL', 'FAKE', 'FAKE', 'FAKE', 'FAKE', 'REAL', 'FAKE',
       'FAKE', 'FAKE', 'FAKE', 'REAL', 'REAL', 'REAL', 'REAL', 'FAKE',
       'FAKE', 'FAKE', 'FAKE', 'REAL', 'FAKE', 'FAKE', 'FAKE', 'FAKE',
       'REAL', 'FAKE', 'REAL', 'FAKE', 'FAKE', 'REAL', 'FAKE', 'REAL',
       'FAKE', 'REAL', 'FAKE', 'REAL', 'FAKE', 'FAKE', 'REAL', 'FAKE',
       'FAKE', 'FAKE', 'REAL', 'REAL', 'FAKE', 'REAL', 'FAKE', 'REAL',
       'FAKE', 'FAKE', 'FAKE', 'REAL', 'REAL', 'FAKE', 'REAL', 'REAL',
       'FAKE', 'FAKE', 'REAL', 'REAL', 'REAL', 'FAKE', 'FAKE', 'FAKE',
       'FAKE', 'REAL', 'FAKE', 'FAKE', 'FAKE', 'REAL', 'FAKE', 'FAKE',
       'FAKE', 'FAKE', 'FAKE', 'REAL', 'FAKE', 'REAL', 'REAL', 'FAKE',
       'FAKE', 'FAKE', 'REAL', 'FAKE', 'REAL', 'FAKE', 'FAKE', 'REAL',
      

In [ ]:
print("Predicción:\n", y_pred)
print("\nEtiquetas reales:\n", y_test.values)

Predicción:
 ['REAL' 'REAL' 'FAKE' 'FAKE' 'REAL' 'FAKE' 'FAKE' 'FAKE' 'FAKE' 'FAKE'
 'REAL' 'REAL' 'FAKE' 'FAKE' 'FAKE' 'REAL' 'REAL' 'REAL' 'REAL' 'FAKE'
 'REAL' 'REAL' 'REAL' 'FAKE' 'REAL' 'REAL' 'FAKE' 'FAKE' 'FAKE' 'FAKE'
 'REAL' 'FAKE' 'FAKE' 'FAKE' 'FAKE' 'REAL' 'REAL' 'REAL' 'REAL' 'FAKE'
 'FAKE' 'FAKE' 'FAKE' 'REAL' 'FAKE' 'FAKE' 'FAKE' 'FAKE' 'REAL' 'FAKE'
 'REAL' 'FAKE' 'FAKE' 'REAL' 'FAKE' 'REAL' 'FAKE' 'REAL' 'FAKE' 'REAL'
 'FAKE' 'FAKE' 'REAL' 'FAKE' 'FAKE' 'FAKE' 'REAL' 'REAL' 'FAKE' 'REAL'
 'FAKE' 'REAL' 'FAKE' 'FAKE' 'FAKE' 'REAL' 'REAL' 'FAKE' 'REAL' 'REAL'
 'FAKE' 'FAKE' 'REAL' 'REAL' 'REAL' 'FAKE' 'FAKE' 'FAKE' 'FAKE' 'REAL'
 'FAKE' 'FAKE' 'FAKE' 'REAL' 'FAKE' 'FAKE' 'FAKE' 'FAKE' 'FAKE' 'REAL'
 'FAKE' 'REAL' 'REAL' 'FAKE' 'FAKE' 'FAKE' 'REAL' 'FAKE' 'REAL' 'FAKE'
 'FAKE' 'REAL' 'FAKE' 'REAL' 'REAL' 'FAKE' 'REAL' 'FAKE' 'FAKE' 'REAL'
 'REAL' 'FAKE' 'REAL' 'FAKE' 'FAKE' 'FAKE' 'REAL' 'FAKE' 'REAL' 'FAKE'
 'FAKE' 'FAKE' 'FAKE' 'FAKE' 'REAL' 'FAKE' 'REAL' 'REAL' 'REAL' 

##### Evaluating the results

In [ ]:
from sklearn.metrics import accuracy_score

print("Accuracy: {:.3f}".format(accuracy_score(y_test, y_pred)))

Accuracy: 0.950


### 9. Increasing the dataset size

The fake news detector already works well with only 1000 news articles. However, Machine Learning algorithms usually perform **better** the more training data they have. Let's train the algorithm with more news.

In [ ]:
# Read 42000 news articles
df_large = df.sample(n=42000, random_state=42)

print("Preprocessing news...")
df_large["text_clean"] = df_large["text"].apply(preprocess_text)
print("Done!")

Preprocessing news...
Done!


In [ ]:
# Use 40000 news articles to train the algorithm and 2000 for testing
X_train, y_train = df_large["text_clean"][:40000], df_large["label"][:40000]
X_test, y_test = df_large["text_clean"][40000:], df_large["label"][40000:]

print(f"Training news articles: {len(X_train)}")
print(f"Test news articles: {len(X_test)}")

Training news articles: 40000
Test news articles: 2000


In [ ]:
vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(X_train)

In [ ]:
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [ ]:
X_test = vectorizer.transform(X_test)

In [ ]:
y_pred = clf.predict(X_test)

In [ ]:
print("Accuracy: {:.3f}".format(accuracy_score(y_test, y_pred)))

Accuracy: 0.987


### 10. Real scenario - Testing with new news created by ourselves

1. Preprocess
2. Vectorize
3. Model prediction
4. Probabilities assigned by the model to each class

In [ ]:
# Define some test news articles
new_news_articles = [
    """DUBAI, Feb 13 (Reuters) - Dubai port giant DP World said on Friday its chairman and chief executive Sultan Ahmed Bin Sulayem had resigned, an announcement that followed mounting pressure over his alleged ties to Jeffrey Epstein.
Bin Sulayem, one of the Middle East's most prominent business figures, is among the highest-profile executives to face scrutiny and be removed from senior roles following the recent release of the Epstein files.""",
    "EXPOSED: Secret documents reveal that the moon landing was filmed in a Hollywood basement by the CIA!!!",
    """WASHINGTON, Feb 12 (Reuters) - An Arizona sheriff is blocking FBI access to key evidence in the investigation into the abduction of U.S. television journalist Savannah Guthrie's mother, impairing its ability to assist in the probe, a U.S. law enforcement official with knowledge of the case told Reuters on Thursday.
The FBI asked Pima County Sheriff Chris Nanos for physical evidence in the case, including a glove and DNA from the home of 84-year-old Nancy Guthrie, to be processed at the FBI's national crime laboratory in Quantico, Virginia, but Nanos has insisted instead on using a private lab in Florida, the official said.""",
    "YOU WONT BELIEVE THIS: Politicians are secretly lizard people controlling the world government!!!"
]

In [ ]:
processed_news = [preprocess_text(n) for n in new_news_articles]
vectorized_news = vectorizer.transform(processed_news)
predictions = clf.predict(vectorized_news)
probabilities = clf.predict_proba(vectorized_news)

# Display the results
for news, pred, prob in zip(new_news_articles, predictions, probabilities):
    print(f"News: {news[:80]}...")
    print(f"  → Prediction: {pred}")
    print(f"  → Probability FAKE: {prob[0]:.2%} | Probability REAL: {prob[1]:.2%}")
    print()

News: DUBAI, Feb 13 (Reuters) - Dubai port giant DP World said on Friday its chairman ...
  → Prediction: REAL
  → Probability FAKE: 5.18% | Probability REAL: 94.82%

News: EXPOSED: Secret documents reveal that the moon landing was filmed in a Hollywood...
  → Prediction: FAKE
  → Probability FAKE: 99.52% | Probability REAL: 0.48%

News: WASHINGTON, Feb 12 (Reuters) - An Arizona sheriff is blocking FBI access to key ...
  → Prediction: REAL
  → Probability FAKE: 0.10% | Probability REAL: 99.90%

News: YOU WONT BELIEVE THIS: Politicians are secretly lizard people controlling the wo...
  → Prediction: FAKE
  → Probability FAKE: 92.76% | Probability REAL: 7.24%

